# PyHEARTS quick demo

A minimal, end-to-end example: simulate a short ECG, run PyHEARTS, and inspect the
beat-by-beat output.

The public `PyHEARTS` analyzer runs the validated 2025 R/P/Gaussian core followed by
the record-level STPQ T detector. It returns two DataFrames:

- **features** — one row per cardiac cycle (peak locations, Gaussian morphology, intervals)
- **cycles** — the segmented per-beat waveform samples

Install the toolbox first (simulation needs the optional `sim` extra):

```bash
pip install "pyhearts[sim]"
```

## 1. Simulate a short ECG

We use NeuroKit2 to create ~12 seconds of clean ECG at 500 Hz (about 13 beats).

In [1]:
import matplotlib.pyplot as plt
import neurokit2 as nk
import numpy as np

import pyhearts
from pyhearts import PyHEARTS

SAMPLING_RATE_HZ = 500.0

ecg = np.asarray(
    nk.ecg_simulate(
        duration=12,
        sampling_rate=int(SAMPLING_RATE_HZ),
        heart_rate=70,
        random_state=7,
    ),
    dtype=float,
)

print(f"PyHEARTS {pyhearts.__version__}")
print(f"signal: {ecg.size} samples ({ecg.size / SAMPLING_RATE_HZ:.1f} s @ {SAMPLING_RATE_HZ:g} Hz)")

PyHEARTS 2.0.0
signal: 6000 samples (12.0 s @ 500 Hz)


## 2. Run PyHEARTS

`species="human"` selects the production human pipeline. We filter the raw trace with
`preprocess_signal`, then extract beat-level features with `analyze_ecg`.

In [2]:
analyzer = PyHEARTS(sampling_rate=SAMPLING_RATE_HZ, species="human")

filtered = analyzer.preprocess_signal(
    ecg,
    highpass_cutoff=0.5,
    lowpass_cutoff=50.0,
    filter_order=4,
    notch_frequency=60.0,
    quality_factor=30.0,
)

features, cycles = analyzer.analyze_ecg(filtered)

print(f"detected {len(features)} cardiac cycles, {features.shape[1]} features each")
print(f"median reconstruction R^2: {features['r_squared'].median():.3f}")

detected 13 cardiac cycles, 136 features each
median reconstruction R^2: 0.977


## 3. Inspect the beat-by-beat table

Each row is one beat. Below are the P/R/T fiducials, the RR interval, and the fit quality.

`T_global_center_idx` is the STPQ T center; `T_gaussian_global_center_idx` is the original
Gaussian T center, kept separately so timing and morphology stay distinguishable.

In [3]:
summary_columns = [
    "P_global_center_idx",
    "R_global_center_idx",
    "T_global_center_idx",
    "T_gaussian_global_center_idx",
    "t_source",
    "RR_interval_ms",
    "r_squared",
]
features[summary_columns].head(8)

,P_global_center_idx,R_global_center_idx,T_global_center_idx,T_gaussian_global_center_idx,t_source,RR_interval_ms,r_squared
cycle_index,,,,,,,
0,346,433,557.0,555,record_stpq_hybrid,NaN,0.974979
1,779,867,989.0,990,record_stpq_hybrid,868.0,0.917361
2,1208,1295,1417.0,1418,record_stpq_hybrid,856.0,0.972838
3,1635,1722,1845.0,1845,record_stpq_hybrid,854.0,0.974511
4,2064,2151,2274.0,2274,record_stpq_hybrid,858.0,0.973604
5,2493,2580,2702.0,2702,record_stpq_hybrid,858.0,0.976780
6,2920,3006,3130.0,3129,record_stpq_hybrid,852.0,0.978629
7,3350,3437,3560.0,3560,record_stpq_hybrid,862.0,0.979942


## 4. Visualize the detected fiducials

Overlay the P, R, and T centers on the filtered ECG.

In [4]:
time_s = np.arange(filtered.size) / SAMPLING_RATE_HZ

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(time_s, filtered, color="#1f4e79", lw=1.0, label="filtered ECG")

for wave, color in (("P", "#27ae60"), ("R", "#c0392b"), ("T", "#8e44ad")):
    idx = features[f"{wave}_global_center_idx"].dropna().astype(int)
    idx = idx[idx < filtered.size]
    ax.scatter(idx / SAMPLING_RATE_HZ, filtered[idx], s=40, color=color, zorder=3, label=f"{wave} peak")

ax.set_title("PyHEARTS detected P / R / T fiducials")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.legend(loc="upper right", ncol=4, frameon=False)
fig.tight_layout()
plt.show()

/var/folders/p9/jlz2tzy55qb2slsp_lst507m0000gn/T/ipykernel_91955/1488914591.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Next steps

- Swap the simulated signal for your own single-lead ECG (a 1-D NumPy array plus its sampling rate).
- Load a two-column monitor export with `pyhearts.load_monitor_csv`, or a WFDB record with
  `pyhearts.load_wfdb_signal` (needs `pip install "pyhearts[wfdb]"`).
- Persist results with `analyzer.save_output("record_id", "results/")`, which writes a features
  CSV alongside a metadata JSON.
- See `examples/intro_overview.ipynb` for a fuller walkthrough and `docs/PRESETS.md` for configuration.